# Resume-Based Early Turnover Prediction (NLP Classification)

## Purpose
Predict early voluntary turnover (within 90 days) using a combination of:
- **Unstructured data**: Resume text vectorized via TF-IDF
- **Structured data**: Candidate attributes (experience, education, source, etc.)

This demonstrates how NLP techniques can augment traditional predictive models by extracting signal from free-text documents.

## Analytical Techniques
- TF-IDF (Term Frequency-Inverse Document Frequency) text vectorization
- Random Over-Sampling (ROS) for class imbalance correction
- Logistic Regression with GridSearchCV hyperparameter tuning
- ROC AUC evaluation, confusion matrix, classification report
- Model serialization (pickle export)

## Steps
1. Load candidate data with resume text and structured features
2. Vectorize resume text using TF-IDF (up to 5,000 features)
3. Merge TF-IDF features with structured candidate attributes
4. Split into train/test sets
5. Apply RandomOverSampler to balance classes in training data
6. Tune Logistic Regression via GridSearchCV
7. Evaluate on held-out test set (ROC AUC, accuracy, confusion matrix)
8. Export trained model

## Interpretation Guide
- **ROC AUC**: Area under the receiver operating characteristic curve (0.5 = random, 1.0 = perfect)
- **Precision**: Of those predicted to turn over, what proportion actually did?
- **Recall**: Of those who actually turned over, what proportion did we catch?
- **Class imbalance**: Early turnover is rare (~10%), so RandomOverSampler creates synthetic minority examples to prevent the model from always predicting "no turnover"
- **TF-IDF features**: Words that are distinctive to turnover-prone candidates may reveal patterns in background or experience gaps

In [ ]:
# ---- CONFIGURATION ----
DATA_PATH = "synthetic_resumes.csv"
TEXT_COLUMN = "resume_text"
TARGET_COLUMN = "outcome_90day_vol"
MAX_TF_IDF_FEATURES = 5000
TEST_SIZE = 0.25
RANDOM_STATE = 42

In [ ]:
# Step 1: Import Libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, accuracy_score, confusion_matrix,
                             classification_report, roc_curve)
from imblearn.over_sampling import RandomOverSampler

import pickle

print("Libraries loaded successfully.")

In [ ]:
# Step 2: Load Data
df = pd.read_csv(DATA_PATH)
print(f"Dataset: {df.shape[0]} candidates, {df.shape[1]} columns")
print(f"\nTarget distribution:")
print(df[TARGET_COLUMN].value_counts())
print(f"\nTurnover rate: {df[TARGET_COLUMN].mean():.1%}")
df.head()

In [ ]:
# Step 3: TF-IDF Vectorization of Resume Text
# TF-IDF converts text into numerical features where each word's weight
# reflects how important it is to a document relative to the entire corpus.
# Words that appear frequently in one resume but rarely across all resumes
# get higher weights.

print(f"Vectorizing resume text (max {MAX_TF_IDF_FEATURES} features)...")

tfidf = TfidfVectorizer(
    max_features=MAX_TF_IDF_FEATURES,
    stop_words='english',
    ngram_range=(1, 2),  # Include unigrams and bigrams
    min_df=2,            # Ignore very rare terms
    max_df=0.95          # Ignore very common terms
)

tfidf_matrix = tfidf.fit_transform(df[TEXT_COLUMN])
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf.get_feature_names_out()
)

print(f"TF-IDF features created: {tfidf_df.shape[1]}")
print(f"\nTop 20 TF-IDF features (by mean weight):")
print(tfidf_df.mean().sort_values(ascending=False).head(20))

In [ ]:
# Step 4: Merge TF-IDF with Structured Features
# Combining text-derived features with traditional candidate attributes
# gives the model both unstructured and structured signal.

structured_cols = [
    'years_experience', 'gpa', 'internal_candidate',
    'referral', 'num_skills_listed'
]

# Encode categorical columns if needed
if 'education_level' in df.columns:
    edu_dummies = pd.get_dummies(df['education_level'], prefix='edu')
    structured_features = pd.concat([df[structured_cols], edu_dummies], axis=1)
else:
    structured_features = df[structured_cols]

if 'source' in df.columns:
    source_dummies = pd.get_dummies(df['source'], prefix='source')
    structured_features = pd.concat([structured_features, source_dummies], axis=1)

# Combine all features
X = pd.concat([tfidf_df, structured_features.reset_index(drop=True)], axis=1)
y = df[TARGET_COLUMN]

print(f"Total feature matrix: {X.shape[0]} rows x {X.shape[1]} features")
print(f"  TF-IDF features: {tfidf_df.shape[1]}")
print(f"  Structured features: {structured_features.shape[1]}")

In [ ]:
# Step 5: Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")
print(f"\nTraining class distribution:")
print(y_train.value_counts())
print(f"\nClass imbalance ratio: {y_train.value_counts()[0] / max(y_train.value_counts()[1], 1):.1f}:1")

In [ ]:
# Step 6: Handle Class Imbalance with Random Over-Sampling
# RandomOverSampler duplicates minority class examples so the model
# doesn't just learn to predict the majority class every time.

ros = RandomOverSampler(random_state=RANDOM_STATE)
X_train_balanced, y_train_balanced = ros.fit_resample(X_train, y_train)

print(f"Before oversampling: {Counter(y_train)}")
print(f"After oversampling:  {Counter(y_train_balanced)}")

In [ ]:
# Step 7: Hyperparameter Tuning with GridSearchCV
# Testing different regularization strengths and solvers
# for Logistic Regression.

param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'lbfgs'],
    'max_iter': [1000]
}

grid_search = GridSearchCV(
    LogisticRegression(random_state=RANDOM_STATE),
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)

print("Running GridSearchCV (5-fold cross-validation)...")
grid_search.fit(X_train_balanced, y_train_balanced)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV ROC AUC: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_

In [ ]:
# Step 8: Evaluate on Test Set
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# Metrics
roc_auc = roc_auc_score(y_test, y_pred_proba)
accuracy = accuracy_score(y_test, y_pred)

print("=" * 50)
print("TEST SET EVALUATION")
print("=" * 50)
print(f"ROC AUC:  {roc_auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred,
                            target_names=['Retained', 'Turnover']))

In [ ]:
# Step 9: Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Retained', 'Turnover'],
            yticklabels=['Retained', 'Turnover'],
            ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, color='darkorange', lw=2,
             label=f'ROC (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--',
             label='Random (AUC = 0.50)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# Step 10: Feature Importance (Top Predictive Words)
# For logistic regression, the coefficient magnitude indicates
# how strongly each feature predicts the outcome.

feature_names = list(X.columns)
coefficients = best_model.coef_[0]

feature_importance = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients,
    'abs_coefficient': np.abs(coefficients)
}).sort_values('abs_coefficient', ascending=False)

print("Top 20 Most Predictive Features:")
print("(Positive = predicts turnover, Negative = predicts retention)")
print(feature_importance.head(20).to_string(index=False))

# Plot top features
top_n = 15
top_features = feature_importance.head(top_n)

plt.figure(figsize=(10, 6))
colors = ['coral' if x > 0 else 'steelblue' for x in top_features['coefficient']]
plt.barh(range(top_n), top_features['coefficient'], color=colors)
plt.yticks(range(top_n), top_features['feature'])
plt.xlabel('Coefficient Value')
plt.title('Top Predictive Features\n(Red = Turnover Risk, Blue = Retention Signal)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Step 11: Export Trained Model
MODEL_OUTPUT_PATH = "turnover_prediction_model.pkl"

with open(MODEL_OUTPUT_PATH, 'wb') as f:
    pickle.dump({
        'model': best_model,
        'tfidf_vectorizer': tfidf,
        'feature_names': feature_names,
        'best_params': grid_search.best_params_,
        'test_roc_auc': roc_auc
    }, f)

print(f"Model exported to: {MODEL_OUTPUT_PATH}")
print(f"Contains: trained model, TF-IDF vectorizer, feature names, best params")